# CSA10 continuation — XE, two GPUs
            **T4 ×2**, Internet ON, Save Version → Save & Run All. One notebook per account;
            both run two GPUs. No API secrets needed for these training notebooks.

            Add Input: `csa-full-roi-karpathy-v134`, `csa-full-xe10-v139`,
            and saved `csa-grounding-audit10k-v150`. Edit mount paths below if necessary.
            No raw images/detector/bank130/new CLIP extraction required; banks are in checkpoint139.
            Exact epoch10 `checkpoint.bin` required, NOT best9 or epoch15.

            XE: only XE.
            Fresh Adam LR1e-5, global32, FP32, same 64,000 events/2,000 updates (~1.28 subset epochs).
            Attention instrumentation identical across arms; original inference attention/beam5/max20.
            Fixed endpoint then VALID5000 CIDEr/BLEU4/ROUGE-L/CHAIR/recall/NLL; no TEST or best selection.
            Weak labels are NOT human-verified; this is an exploratory pilot.

            V2 fixes missing author-generator source required by the packaged AST test.
            The old run stopped before training; leave RESUME_ROOT empty for that failure.
            V3 checks CUDA build, availability and exactly two GPUs BEFORE staging/installing
            dependencies. Select T4 x2 explicitly after import; metadata alone does not guarantee
            accelerator allocation. A CPU-only environment must be restarted as a GPU session.
            Output has a fresh _v3 name to preserve the failed run. Same data/weights/loss/2000updates;
            reuse existing label Dataset, do not rerun its CPU save notebook.

            Parent4h stop cap, not ETA/billing guarantee. Checkpoints every500updates plus endpoint;
            preserve whole output for same-arm resume. First new session still initializes weights
            from CSA10, not fresh random weights. Both CPU cache staging and user-run neural/DDP
            preflights must pass; assistant has not run models locally.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
ARM='XE'
AVAILABLE_GPU_HOURS=20 # Last user report; cap below is not a billing estimate.
WALL_HOURS=4
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
AUDIT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-grounding-audit10k-v150')
CHECKPOINT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-xe10-v139')
RESUME_ROOT='' # Exact prior SAME-ARM output folder containing checkpoint.bin + receipt; otherwise fresh continuation.
OUTPUT=Path('/kaggle/working/csa_grounding_pilot_'+ARM+'_v3')
for p in (FULL_ROOT,AUDIT_ROOT,CHECKPOINT_ROOT):
    if not p.is_dir():raise FileNotFoundError('Attach all three input Datasets before GPU: '+str(p))
GPU_ENV=json.loads(subprocess.check_output([sys.executable,'-c','import json,torch\nprint(json.dumps(dict(torch_version=torch.__version__,cuda_built=torch.backends.cuda.is_built(),cuda_version=torch.version.cuda,available=torch.cuda.is_available(),device_count=torch.cuda.device_count())))\n'],text=True,timeout=60))
print('GPU environment:',GPU_ENV,flush=True)
if not GPU_ENV['cuda_built']:
    raise RuntimeError('CPU-only PyTorch environment. Select Settings > Accelerator > GPU T4 x2 and restart the Kaggle session. If already selected, use a fresh GPU session; do not reinstall torch blindly.')
if not GPU_ENV['available'] or GPU_ENV['device_count']!=2:
    raise RuntimeError('Need exactly two usable CUDA GPUs. Select T4 x2, restart, and check GPU quota/session allocation before running again.')
if OUTPUT.exists():raise RuntimeError('Preserve prior output; use fresh runtime/output path for resume.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<8*1024**3:raise RuntimeError('Need8GiB free for readonly stage and atomic checkpoints.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa-grounding-'+ARM+'-'));STAGE=SOURCE/'stage';LOG=OUTPUT/'notebook.log'
os.environ['OMP_NUM_THREADS']='1'

COMMIT='9825f481711e8fed40efa30189848df829b40f16'
HASHES={'grounding_pilot_data.py': '175ebdb3307fa76847b94746122e7281ffb7789a55d575527e6dcf70b13db392', 'grounding_model.py': 'dbda9b5fc6cf5eb9116d87b692db7516979e0e113b993c0b0fc2f5b8402110e2', 'run_grounding_pilot.py': 'b0e73dbf3c87e66c7144fc8782f5b7142ed4bd5390a456ff8d8c224c7c848280', 'test_grounding_model.py': '2268dd1d5491bd8cea9788f3d3f9128dae4bc02442d31ace459ea987af85db68', 'test_grounding_pilot_data.py': '52c6def55050073fe0e2d079bac95ce9016e3539e911ae933fec99fcdf80c0ca', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'grounding_audit_v2_review_text/summary.json': 'a044c0d2e2d88deb1926b88311a6b289f48e46c9493c0c8ebf4a5bc2230b8cb6', 'GROUNDING_PILOT_PROTOCOL.md': 'dc9f88709776eb9dec35e4c8da80e3b14240b7691727160407eaa3a50b2aaa52', 'package_grounding150.py': '92b84ab703a5de7bb9806f84ebb3010bf50a3e273d0b93061d9e6db54649c1fe', 'make_grounding_pilot_notebooks.py': '297c30a86dbd2145c32dc4e071004829d878a9ffce0b3b78a3ff0bcbcc6ff826', 'make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054'}
for name,expected in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned source:'+name)
    p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)

(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')
(OUTPUT/'gpu_environment.json').write_text(json.dumps(GPU_ENV,indent=2),encoding='utf-8')
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair'
cfg=dict(arm=ARM,full_root=str(FULL_ROOT),audit_root=str(AUDIT_ROOT),checkpoint_root=str(CHECKPOINT_ROOT),output=str(OUTPUT),stage=str(STAGE),chair=str(CHAIR),resume_root=RESUME_ROOT,worker_seconds=9000)
CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
print('Prepared',ARM,'source',COMMIT,'fixed2000updates; VALID only.',flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Grounding pilot active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False;parent_deadline=time.monotonic()+WALL_HOURS*3600
try:
    run([sys.executable,'-u',SOURCE/'test_grounding_pilot_data.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    run([sys.executable,'-u',SOURCE/'run_grounding_pilot.py','prepare','--config',CONFIG],timeout=2400)
    run([sys.executable,'-u',SOURCE/'run_grounding_pilot.py','metric_preflight','--config',CONFIG],timeout=300)
    run([sys.executable,'-u',SOURCE/'test_grounding_model.py'],timeout=180)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'test_grounding_model.py','--ddp'],timeout=300)
    remaining=parent_deadline-time.monotonic()
    if remaining<1800:raise TimeoutError('Insufficient worker + scoring margin; no automatic training')
    cfg['worker_seconds']=min(10800,remaining-1200);CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    run([*launch,SOURCE/'run_grounding_pilot.py','worker','--config',CONFIG],timeout=cfg['worker_seconds']+60)
    run([sys.executable,'-u',SOURCE/'run_grounding_pilot.py','score','--config',CONFIG],timeout=min(1200,parent_deadline-time.monotonic()))
    complete=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))['status']=='CSA_GROUNDING_PILOT_COMPLETE'
    if not complete:raise RuntimeError('Pilot incomplete')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,updates_requested=2000,test_used=False,labels_human_verified=False)),encoding='utf-8')
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin' if (OUTPUT/'checkpoint.bin').is_file() else 'No checkpoint yet',flush=True)
    print('COMPLETE endpoint2000 + VALID5000' if complete else 'INCOMPLETE: preserve output/logs; no automatic restart.',flush=True)
